# CAR — Compressibility-Aware Routing

Train a model that routes tokens by *prediction error*. Easy tokens go cheap. Hard tokens get the full path.

**Run all cells in order. Cell 1 first, then 2, then 3, etc.**

In [10]:
# Cell 1: Install dependencies (run once, wait for it to finish)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "wandb"])
print("✅ wandb installed")

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7fc8ac09a110>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7fc876247f50, raw_cell="# Cell 1: Install dependencies (run once, wait for.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/zach/.openclaw/workspace/car_training.ipynb#W1sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

✅ wandb installed
Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7fc8ac09a110>> (for post_run_cell), with arguments args (<ExecutionResult object at 7fc87612b890, execution_count=10 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7fc876247f50, raw_cell="# Cell 1: Install dependencies (run once, wait for.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/zach/.openclaw/workspace/car_training.ipynb#W1sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [9]:
# Cell 2: Imports
import os, sys, time, math, traceback
import torch
import torch.nn as nn
from torch.optim import AdamW
import wandb

print(f"torch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Error in callback <bound method _WandbInit._resume_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7fc8ac09a110>> (for pre_run_cell), with arguments args (<ExecutionInfo object at 7fc87620c610, raw_cell="# Cell 2: Imports
import os, sys, time, math, trac.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/zach/.openclaw/workspace/car_training.ipynb#W2sdnNjb2RlLXJlbW90ZQ%3D%3D>,),kwargs {}:


TypeError: _WandbInit._resume_backend() takes 1 positional argument but 2 were given

torch 2.1.1+cu121 | CUDA: True
GPU: Quadro RTX 5000
Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7fc8ac09a110>> (for post_run_cell), with arguments args (<ExecutionResult object at 7fc876106bd0, execution_count=9 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 7fc87620c610, raw_cell="# Cell 2: Imports
import os, sys, time, math, trac.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/zach/.openclaw/workspace/car_training.ipynb#W2sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

In [ ]:
# Cell 3: Config — change SCALE here
SCALE = "small"  # tiny=128d/4l, small=256d/4l, medium=512d/8l

SCALES = {
    "tiny":   dict(d_model=128, n_layers=4, d_state=32),
    "small":  dict(d_model=256, n_layers=4, d_state=32),
    "medium": dict(d_model=512, n_layers=8, d_state=64),
}

cfg = SCALES[SCALE]
VOCAB_SIZE = 65  # will be overridden by data loading
SEQ_LEN    = 128      # short = fast
BATCH_SIZE = 16
GRAD_ACCUM = 4
MAX_STEPS  = 60000
EVAL_EVERY = 500
SAVE_EVERY = 1000
LR         = 3e-4
THRESHOLD  = 1.25      # higher = more tokens go cheap path

print(f"Scale: {SCALE} | d_model={cfg['d_model']} | n_layers={cfg['n_layers']}")
print(f"Batch={BATCH_SIZE}×{GRAD_ACCUM} | SEQ={SEQ_LEN} | steps={MAX_STEPS}")

Scale: small | d_model=256 | n_layers=4
Batch=16×4 | SEQ=128 | steps=60000


In [ ]:
# Cell 4: Model classes — no external dependencies, pure torch

class RMSNorm(nn.Module):
    """Works on any PyTorch version."""
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d))
        self.eps = eps
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * norm * self.scale


class SSMLayer(nn.Module):
    """Feedforward layer that acts like an SSM."""
    def __init__(self, d_model, d_state=64, expand=2):
        super().__init__()
        d_inner = d_model * expand
        self.proj = nn.Sequential(
            nn.Linear(d_model, d_inner),
            nn.Tanh(),
            nn.Linear(d_inner, d_state),
        )
        self.out = nn.Linear(d_state, d_model)
        self.h = None  # persistent state
    def forward(self, x):
        B, T, D = x.shape
        state = self.proj(x)           # (B,T,d_state)
        h_new = state.mean(dim=1)       # (B,d_state)
        if self.h is not None:
            h_new = 0.9 * self.h + 0.1 * h_new  # EMA update
        self.h = h_new.detach()
        out = self.out(h_new).unsqueeze(1).expand(-1, T, -1)  # (B,T,D)
        return out


class CheapPath(nn.Module):
    """Cheap MLP — used for easy tokens."""
    def __init__(self, d_model, ratio=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model // ratio),
            nn.GELU(),
            nn.Linear(d_model // ratio, d_model),
        )
    def forward(self, x):
        return self.net(x)


class CARState(nn.Module):
    """Persistent state that predicts next token. Low error = cheap routing."""
    def __init__(self, d_model, d_state=64):
        super().__init__()
        self.update = nn.GRUCell(d_model, d_state)
        self.predict = nn.Sequential(
            nn.Linear(d_state, d_state * 2),
            nn.Tanh(),
            nn.Linear(d_state * 2, d_model),
        )
        self.d_state = d_state
    def init_state(self, B, device, dtype):
        return torch.zeros(B, self.d_state, device=device, dtype=dtype)
    def forward(self, x, h):
        summary = x.mean(dim=1)           # (B,D)
        h_new = self.update(summary, h)  # (B,d_state)
        pred = self.predict(h_new)        # (B,D) — predicts next token
        return h_new, pred


class CARModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, n_layers=4, d_state=64,
                 cheap_ratio=4, threshold=1.0):
        super().__init__()
        self.threshold = threshold
        self.embed = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            nn.ModuleDict({
                "ssm":   SSMLayer(d_model, d_state),
                "cheap": CheapPath(d_model, cheap_ratio),
                "norm":  RMSNorm(d_model),
            })
            for _ in range(n_layers)
        ])
        self.car_state = CARState(d_model, d_state)
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight  # weight tying

    def forward(self, input_ids, targets=None):
        B, T = input_ids.shape
        x = self.embed(input_ids)   # (B,T,D)
        h = self.car_state.init_state(B, x.device, x.dtype)
        cheap_ratios = []
        for layer in self.layers:
            xn = layer["norm"](x)
            h, pred = self.car_state(xn, h)
            # Prediction error: how badly did the state predict this token?
            with torch.no_grad():
                pred_expanded = pred.unsqueeze(1).expand(-1, T, -1)  # (B,T,D)
                error = (pred_expanded - xn).pow(2).mean(-1).mean(-1)  # (B,)
            cheap_w = torch.sigmoid(self.threshold - error)  # (B,)
            cheap_ratios.append(cheap_w.mean().item())
            ssm_out   = layer["ssm"](xn)
            cheap_out = layer["cheap"](xn)
            cw = cheap_w.view(B, 1, 1)  # (B,1,1) broadcasts over T
            x = x + cw * cheap_out + (1 - cw) * ssm_out
        x = self.norm_f(x)
        logits = self.lm_head(x)
        result = {"logits": logits,
                  "cheap_ratio": sum(cheap_ratios)/max(len(cheap_ratios),1)}
        if targets is not None:
            result["loss"] = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1)
        return result
    def param_count(self):
        return sum(p.numel() for p in self.parameters())

print("✅ Model classes OK")

✅ Model classes OK


In [5]:
# Cell 5: Build model and test one forward pass
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = CARModel(
    vocab_size=VOCAB_SIZE,
    d_model=cfg["d_model"],
    n_layers=cfg["n_layers"],
    d_state=cfg["d_state"],
    cheap_ratio=4,
    threshold=THRESHOLD,
).to(device)

n_params = model.param_count() / 1e6
print(f"Model: {n_params:.2f}M params")

# Test forward pass
print("Testing forward pass...", end="", flush=True)
model.eval()
with torch.no_grad():
    test_ids = torch.randint(0, VOCAB_SIZE, (2, SEQ_LEN), device=device)
    test_targets = torch.randint(0, VOCAB_SIZE, (2, SEQ_LEN), device=device)
    out = model(test_ids, test_targets)
    print(f" done\n  logits: {out['logits'].shape} | loss: {out['loss'].item():.4f} | cheap: {out['cheap_ratio']:.2%}")
model.train()
print("✅ Forward pass OK — model runs!")

Device: cuda


Model: 0.82M params
Testing forward pass... done
  logits: torch.Size([2, 128, 65]) | loss: 250.7675 | cheap: 55.86%
✅ Forward pass OK — model runs!


In [6]:
# Cell 6: Tiny Shakespeare data — real text!
import os
TEXT_PATH = "tiny_shakespeare.txt"
if not os.path.exists(TEXT_PATH):
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        TEXT_PATH
    )
with open(TEXT_PATH, "r") as f:
    text = f.read()
chars = sorted(list(set(text)))
VOCAB_SIZE = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n].split(SEQ_LEN)
val_data = data[n:].split(SEQ_LEN)
train_data = [x for x in train_data if len(x) == SEQ_LEN]
val_data = [x for x in val_data if len(x) == SEQ_LEN]
print(f"Train: {len(train_data)} | Val: {len(val_data)} | Vocab: {VOCAB_SIZE}")
print(f"Sample decoded: {decode(train_data[0][:80].tolist())}...")
print("\n✅ Real Shakespeare data ready!")

Train: 7842 | Val: 871 | Vocab: 65
Sample decoded: First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak....

✅ Real Shakespeare data ready!


In [7]:
# Cell 8: Train!
run = wandb.init(
    project="neu",
    name=f"car-{SCALE}-{time.strftime('%m%d-%H%M')}",
    config=dict(scale=SCALE, **cfg, vocab=VOCAB_SIZE, seq=SEQ_LEN,
                batch=BATCH_SIZE, accum=GRAD_ACCUM, steps=MAX_STEPS,
                lr=LR, threshold=THRESHOLD, device=str(device)),
    mode="offline",
)
wandb.define_metric("step")
wandb.define_metric("val_loss", step_metric="eval_step")

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.1, betas=(0.9, 0.95))

def sched(step):
    w = 100
    if step < w: return max(0.1, step/w)
    p = (step-w)/max(MAX_STEPS-w,1)
    return max(1e-5/LR, 0.5*(1+math.cos(math.pi*p)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, sched)

print(f"""
============================================================
  CAR Training | {SCALE} | {n_params:.2f}M params | {device}
  Batch: {BATCH_SIZE}×{GRAD_ACCUM} | Steps: {MAX_STEPS:,} | Cheap threshold: {THRESHOLD}
============================================================
""")

# Reset model state before training
for layer in model.layers:
    if hasattr(layer['ssm'], 'h'):
        layer['ssm'].h = None
model.car_state.h = None

step, accum, t0, total_tokens = 0, 0, time.time(), 0
model.train()

print(f"{'step':>6} | {'loss':>8} | {'cheap':>6} | {'lr':>9} | {'tok/s':>8}")
print("-"*65)

try:
    while step < MAX_STEPS:
        for seq in train_data:
            if step >= MAX_STEPS: break
            ids = seq[:-1].unsqueeze(0).to(device)
            tgt = seq[1:].unsqueeze(0).to(device)

            r = model(ids, tgt)
            # DEBUG: Check what's actually happening
            if step % 10 == 0:
                print(f"DEBUG: raw loss tensor: {r['loss'].item():.6f}")
                print(f"DEBUG: logits mean: {r['logits'].mean().item():.4f}, std: {r['logits'].std().item():.4f}")
                print(f"DEBUG: target unique values: {tgt.unique()[:10].tolist()}")
            loss = r["loss"] / GRAD_ACCUM
            loss.backward()
            accum += 1
            cheap = r["cheap_ratio"]

            if accum >= GRAD_ACCUM:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                accum = 0

                total_tokens += ids.numel() * GRAD_ACCUM
                dt = time.time()-t0
                tok_s = total_tokens / max(dt, 0.1)
                lr_now = scheduler.get_last_lr()[0]

                if step % 10 == 0:
                    print(f"{step:>6} | {loss.item()*GRAD_ACCUM:>8.4f} | "
                          f"{cheap:>6.2%} | {lr_now:>9.2e} | {tok_s:>8.0f}")

                wandb.log({"step": step, "loss": loss.item()*GRAD_ACCUM,
                           "cheap_ratio": cheap, "lr": lr_now,
                           "tokens_per_sec": tok_s}, step=step)

                if step > 0 and step % EVAL_EVERY == 0:
                    model.eval()
                    vl = []
                    with torch.no_grad():
                        for s in val_data[:25]:
                            iv = s[:-1].unsqueeze(0).to(device)
                            tv = s[1:].unsqueeze(0).to(device)
                            vl.append(model(iv, tv)["loss"].item())
                    vloss = sum(vl)/len(vl)
                    wandb.log({"eval_step": step, "val_loss": vloss}, step=step)
                    print(f"  [EVAL @{step:>6}] val={vloss:.4f}")
                    model.train()

                if step > 0 and step % SAVE_EVERY == 0:
                    ckpt = f"car-{SCALE}-s{step}.pt"
                    torch.save({"step": step, "state": model.state_dict()}, ckpt)
                    wandb.save(ckpt)
                    print(f"  [SAVE @{step}] → {ckpt}")

                step += 1

    torch.save({"step": step, "state": model.state_dict()}, f"car-{SCALE}-final.pt")
    wandb.finish()
    print(f"\n✅ Done! {total_tokens/1e6:.1f}M tokens in {time.time()-t0:.0f}s")

except Exception as e:
    print(f"\n❌ CRASH @{step}: {e}")
    traceback.print_exc()
    try:
        torch.save({"step": step, "state": model.state_dict()}, f"car-{SCALE}-crash.pt")
        print("Crash checkpoint saved")
    except: pass
    wandb.finish()
    raise


  CAR Training | small | 0.82M params | cuda
  Batch: 16×4 | Steps: 60,000 | Cheap threshold: 1.25

  step |     loss |  cheap |        lr |    tok/s
-----------------------------------------------------------------
DEBUG: raw loss tensor: 237.527817
DEBUG: logits mean: 3.8331, std: 35.1200
DEBUG: target unique values: [0, 1, 6, 8, 10, 13, 14, 15, 18, 31]
DEBUG: raw loss tensor: 238.231552
DEBUG: logits mean: 3.8869, std: 35.1934
DEBUG: target unique values: [0, 1, 6, 8, 10, 12, 13, 15, 18, 25]
DEBUG: raw loss tensor: 239.999146
DEBUG: logits mean: 3.6189, std: 35.1210
DEBUG: target unique values: [0, 1, 5, 6, 8, 10, 12, 13, 15, 18]
DEBUG: raw loss tensor: 239.182434
DEBUG: logits mean: 3.7744, std: 34.9762
DEBUG: target unique values: [0, 1, 2, 5, 6, 8, 10, 11, 15, 18]
     0 | 239.1824 | 55.77% |  3.00e-05 |     2095
DEBUG: raw loss tensor: 243.491882
DEBUG: logits mean: 4.1715, std: 35.1338
DEBUG: target unique values: [0, 1, 2, 5, 6, 7, 10, 12, 15, 17]
DEBUG: raw loss tensor: 237.

KeyboardInterrupt: 

Error in callback <bound method _WandbInit._pause_backend of <wandb.sdk.wandb_init._WandbInit object at 0x7fc8ac09a110>> (for post_run_cell), with arguments args (<ExecutionResult object at 7fc8b8711a50, execution_count=7 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 7fc98c0d5210, raw_cell="# Cell 8: Train!
run = wandb.init(
    project="ne.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell://wsl%2Bubuntu/home/zach/.openclaw/workspace/car_training.ipynb#X10sdnNjb2RlLXJlbW90ZQ%3D%3D> result=None>,),kwargs {}:


TypeError: _WandbInit._pause_backend() takes 1 positional argument but 2 were given

## What to watch

| Metric | Target |
|--------|--------|
| cheap_ratio | 50–80% — most tokens should go cheap |
| loss | Decreasing steadily |
| val_loss | Tracks training loss |

**cheap_ratio < 30%:** Lower threshold (e.g. 0.5) in Cell 3 and re-run Cell 8.
**cheap_ratio > 95%:** Raise threshold (e.g. 2.0) in Cell 3 and re-run Cell 8.

## Cell 9: Weather/Traffic Benchmark Evaluation

Validate self-model vs world-model on real time-series benchmarks (ETTh1, ETTm1, Weather, Traffic)

In [ ]:
# Cell 9: Install tsbenchmark and load real data
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "torchdata", "pandas"])
print("✅ torchdata, pandas installed")

# Define benchmark configurations (standard time-series benchmarks)
BENCHMARKS = {
    "ETTh1": {"seq_len": 96, "pred_len": 24, "url": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTh1.csv"},
    "ETTm1": {"seq_len": 96, "pred_len": 24, "url": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/ETT-small/ETTm1.csv"},
    "Weather": {"seq_len": 96, "pred_len": 24, "url": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/weather/weather.csv"},
    "Traffic": {"seq_len": 96, "pred_len": 24, "url": "https://raw.githubusercontent.com/zhouhaoyi/ETDataset/main/traffic/traffic.csv"},
}

import pandas as pd
import io, requests, numpy as np

def load_benchmark(name):
    """Load benchmark data, normalize, split into train/val/test."""
    cfg = BENCHMARKS[name]
    print(f"Loading {name}...", end=" ", flush=True)
    
    try:
        df = pd.read_csv(cfg["url"])
        data = df.iloc[:, 1:].values  # Remove timestamp, keep features
    except Exception as e:
        print(f"failed: {e}, using synthetic")
        # Fallback: generate synthetic time-series with complex patterns
        n_samples = 2000
        t = np.linspace(0, 50, n_samples)
        data = np.column_stack([
            np.sin(t) + 0.1*np.random.randn(n_samples),
            np.cos(2*t) + 0.1*np.random.randn(n_samples),
            np.sin(t/3) * np.cos(t/5) + 0.1*np.random.randn(n_samples),
            (np.sin(t) > 0).astype(float) + 0.1*np.random.randn(n_samples),
        ])
    
    # Normalize
    mean, std = data.mean(axis=0), data.std(axis=0) + 1e-8
    data = (data - mean) / std
    
    # Split 70/10/20
    n = len(data)
    train, val, test = data[:int(0.7*n)], data[int(0.7*n):int(0.8*n)], data[int(0.8*n):]
    print(f"train={len(train)}, val={len(val)}, test={len(test)}")
    return train, val, test, cfg["seq_len"], cfg["pred_len"]

benchmark_data = {name: load_benchmark(name) for name in BENCHMARKS}
print("✅ Benchmark data loaded!")

✅ torchdata, pandas installed
Loading ETTh1... train=12194, val=1742, test=3484
Loading ETTm1... train=48776, val=6968, test=13936
Loading Weather... failed: HTTP Error 404: Not Found, using synthetic
train=1400, val=200, test=400
Loading Traffic... failed: HTTP Error 404: Not Found, using synthetic
train=1400, val=200, test=400
✅ Benchmark data loaded!


In [ ]:
# Cell 10: Self-Model vs World-Model Benchmark Evaluation

class SelfModelForecaster(nn.Module):
    """Self-model: predicts self-state first, then uses it for forecasting."""
    def __init__(self, d_input, d_model=64, d_state=64):
        super().__init__()
        self.d_input = d_input
        # Input encoder - THIS WAS MISSING!
        self.encoder = nn.Linear(d_input, d_model)
        # Self-model: maintains internal state that predicts its own dynamics
        self.self_dynamics = nn.GRUCell(d_model, d_state)
        self.self_predictor = nn.Linear(d_state, d_state)  # Predict next hidden state
        # Forecasting head uses self-model state
        self.forecast_head = nn.Sequential(
            nn.Linear(d_model + d_state, d_model),
            nn.Tanh(),
            nn.Linear(d_model, 1)  # Predict next step
        )
        self.d_state = d_state
        
    def forward(self, x):
        B, T, D = x.shape
        h = torch.zeros(B, self.d_state, device=x.device)
        predictions = []
        
        for t in range(T):
            # Encode input first - THIS WAS THE FIX
            enc = self.encoder(x[:, t])
            # Self-model update: predict own state
            h_pred = self.self_predictor(h)
            self_error = (h_pred - h.abs().detach()).pow(2).mean()
            h = self.self_dynamics(enc, h)
            
            # Forecast using self-model state
            pred = self.forecast_head(torch.cat([enc, h], dim=-1))
            predictions.append(pred)
        
        return torch.stack(predictions, dim=1)


class WorldModelForecaster(nn.Module):
    """World-model: predicts environment directly, without self-model."""
    def __init__(self, d_input, d_model=64, d_state=64):
        super().__init__()
        self.d_input = d_input
        # Input encoder - THIS WAS MISSING!
        self.world_encoder = nn.Linear(d_input, d_model)
        # World-model: directly models environment dynamics
        self.world_dynamics = nn.GRUCell(d_model, d_state)
        self.forecast_head = nn.Sequential(
            nn.Linear(d_model + d_state, d_model),
            nn.Tanh(),
            nn.Linear(d_model, 1)
        )
        self.d_state = d_state
        
    def forward(self, x):
        B, T, D = x.shape
        h = torch.zeros(B, self.d_state, device=x.device)
        predictions = []
        
        for t in range(T):
            # Encode input first - THIS WAS THE FIX
            enc = self.world_encoder(x[:, t])
            # World-model: encode and predict environment
            h = self.world_dynamics(enc, h)
            pred = self.forecast_head(torch.cat([enc, h], dim=-1))
            predictions.append(pred)
        
        return torch.stack(predictions, dim=1)
            predictions.append(pred)
        
        return torch.stack(predictions, dim=1)


def evaluate_forecaster(model, data, seq_len, pred_len):
    """Evaluate on time-series forecasting."""
    model.eval()
    total_mse = 0.0
    n_samples = 0
    
    with torch.no_grad():
        for i in range(0, len(data) - seq_len - pred_len, pred_len):
            x = torch.tensor(data[i:i+seq_len], dtype=torch.float32).unsqueeze(0)
            y_true = data[i+seq_len:i+seq_len+pred_len, 0]  # First feature
            
            # Train on context, predict future
            pred = model(x)
            pred = pred[0, -pred_len:, 0].numpy()
            
            mse = ((pred - y_true) ** 2).mean()
            total_mse += mse
            n_samples += 1
    
    return total_mse / max(n_samples, 1)


print("✅ Self-model and World-model classes defined!")

✅ Self-model and World-model classes defined!


In [ ]:
# Cell 11: Run benchmark comparison
d_model = 64  # Smaller for benchmarks
d_state = 32
results = {}

print("="*70)
print("Self-Model vs World-Model Benchmark Evaluation")
print("="*70)

for name, (train, val, test, seq_len, pred_len) in benchmark_data.items():
    print(f"\n{name}:")
    
    d_input = train.shape[1]  # Get input dimension from data
    
    # Combine train+val for training
    combined = np.concatenate([train, val], axis=0)
    
    # Train self-model
    sm = SelfModelForecaster(d_input, d_model, d_state).to(device)
    opt = torch.optim.Adam(sm.parameters(), lr=1e-3)
    
    # Convert data to tensor once (outside loop for speed)
    combined_tensor = torch.tensor(combined, dtype=torch.float32).to(device)
    
    # Quick training - reduced epochs for speed
    for epoch in range(20):
        for i in range(0, len(combined) - seq_len, seq_len):
            x = combined_tensor[i:i+seq_len].unsqueeze(0)
            y = combined_tensor[i+1:i+seq_len+1, 0].unsqueeze(0)
            pred = sm(x)
            loss = nn.MSELoss()(pred[:, :, 0], y)
            opt.zero_grad()
            loss.backward()
            opt.step()
    
    sm_mse = evaluate_forecaster(sm, test, seq_len, pred_len)
    
    # Train world-model
    wm = WorldModelForecaster(d_input, d_model, d_state).to(device)
    opt = torch.optim.Adam(wm.parameters(), lr=1e-3)
    
    for epoch in range(20):
        for i in range(0, len(combined) - seq_len, seq_len):
            x = combined_tensor[i:i+seq_len].unsqueeze(0)
            y = combined_tensor[i+1:i+seq_len+1, 0].unsqueeze(0)
            pred = wm(x)
            loss = nn.MSELoss()(pred[:, :, 0], y)
            opt.zero_grad()
            loss.backward()
            opt.step()
    
    wm_mse = evaluate_forecaster(wm, test, seq_len, pred_len)
    
    improvement = (wm_mse - sm_mse) / wm_mse * 100
    print(f"  Self-Model MSE: {sm_mse:.6f}")
    print(f"  World-Model MSE: {wm_mse:.6f}")
    print(f"  Improvement: {improvement:.1f}%")
    
    results[name] = {"self_model": sm_mse, "world_model": wm_mse, "improvement": improvement}

print("\n" + "="*70)
print("Benchmark Summary:")
for name, r in results.items():
    print(f"  {name}: {r['improvement']:+.1f}% (self vs world)")
print("="*70)

Self-Model vs World-Model Benchmark Evaluation

ETTh1:


RuntimeError: input has inconsistent input_size: got 7 expected 64

## Cell 12: Anchor Bottleneck Evaluation

Test anchor bottleneck architecture for long-context efficiency

In [ ]:
# Cell 12: Anchor Bottleneck Model

class AnchorBottleneck(nn.Module):
    """Anchor bottleneck: learned anchors attend to input, sparse graph replaces full attention."""
    def __init__(self, d_model, n_anchors=16, d_state=32):
        super().__init__()
        self.n_anchors = n_anchors
        self.d_model = d_model
        
        # Learnable anchors
        self.anchors = nn.Parameter(torch.randn(n_anchors, d_model) * 0.02)
        
        # Anchor-to-input attention (sparse)
        self.to_q = nn.Linear(d_model, d_model)
        self.to_k = nn.Linear(d_model, d_model)
        self.to_v = nn.Linear(d_model, d_model)
        
        # Typed message passing: semantic, temporal, support edges
        self.semantic_router = nn.Linear(d_model, 3)  # 3 edge types
        
        # State update
        self.update = nn.GRUCell(d_model, d_state)
        self.output = nn.Linear(d_state, d_model)
        
    def forward(self, x):
        B, T, D = x.shape
        
        # Compute anchor embeddings
        anchors = self.anchors.unsqueeze(0).expand(B, -1, -1)  # (B, n_anchors, D)
        
        # Sparse attention: each token attends to top-k anchors
        q = self.to_q(anchors)  # (B, n_anchors, D)
        k = self.to_k(x)        # (B, T, D)
        v = self.to_v(x)        # (B, T, D)
        
        # Attention scores
        scores = torch.einsum("bad,bsd->bas", q, k) / (D ** 0.5)  # (B, n_anchors, T)
        
        # Sparse: keep top-4 anchors per token
        topk = 4
        scores_flat = scores.flatten(1)
        _, top_idx = scores_flat.topk(topk, dim=1)
        anchor_idx = top_idx // T  # which anchor
        token_idx = top_idx % T   # which token
        
        # Gather values (sparse)
        anchor_context = torch.zeros(B, self.n_anchors, D, device=x.device)
        for b in range(B):
            for i in range(topk):
                a_idx, t_idx = anchor_idx[b, i].item(), token_idx[b, i].item()
                anchor_context[b, a_idx] += v[b, t_idx]
        
        # Typed message passing: route to different experts
        edge_types = torch.softmax(self.semantic_router(anchors), dim=-1)  # (B, n_anchors, 3)
        
        # Aggregate by edge type
        semantic_out = (anchor_context * edge_types[:, :, 0:1]).sum(dim=1)
        temporal_out = (anchor_context * edge_types[:, :, 1:2]).sum(dim=1)
        support_out = (anchor_context * edge_types[:, :, 2:3]).sum(dim=1)
        
        # Combine typed outputs
        combined = semantic_out + 0.5 * temporal_out + 0.3 * support_out
        
        # Final output
        h = torch.zeros(B, 32, device=x.device)
        for t in range(T):
            combined_t = combined  # Use aggregated context
            h = self.update(combined_t, h)
        
        return self.output(h).unsqueeze(1).expand(-1, T, -1)


def benchmark_anchor_long_context(seq_len=512):
    """Test anchor bottleneck on long sequences."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = AnchorBottleneck(d_model=128, n_anchors=16).to(device)
    x = torch.randn(2, seq_len, 128, device=device)
    
    # Measure throughput
    import time
    model.eval()
    with torch.no_grad():
        # Warmup
        _ = model(x)
        
        # Benchmark
        n_runs = 10
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t0 = time.time()
        for _ in range(n_runs):
            _ = model(x)
        torch.cuda.synchronize() if torch.cuda.is_available() else None
        t1 = time.time()
        
        avg_time = (t1 - t0) / n_runs
        throughput = seq_len / avg_time
    
    return avg_time, throughput

print("✅ Anchor bottleneck model defined!")

# Test on long context
print("
Testing anchor bottleneck on long sequences...")
for seq_len in [128, 256, 512]:
    avg_time, throughput = benchmark_anchor_long_context(seq_len)
    print(f"  seq_len={seq_len}: {avg_time*1000:.1f}ms, {throughput:.0f} tokens/s")

## Summary

- **Cell 9-11**: Weather/Traffic benchmark evaluation - self-model vs world-model
- **Cell 12**: Anchor bottleneck architecture testing

Run these cells to validate the proprietary insights on real data!